In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

In [3]:
df_AID_base = pd.read_excel("datasets/Airport_IATA_delays_airline_reported.xlsx")
df_AT = pd.read_excel("datasets/AirportTraffic.xlsx")

In [4]:
df_top20 = (
    df_AT.groupby("APT_ICAO")[["FLT_TOT_1", "FLT_DEP_1", "FLT_ARR_1"]] #Group by airport code
    .sum().sort_values(by="FLT_TOT_1",ascending=False) #Sum the values for each code of the 3 columns indicated
    .head(20).reset_index()) #Change "20" to change the number of airports analysed

# Adding airport's city name and state from original dataset
df_top20 = (df_top20.merge(df_AT[["APT_ICAO", "APT_NAME", "STATE_NAME"]]
                           .drop_duplicates(), on="APT_ICAO", how="left"))
airports_code_list = df_top20["APT_ICAO"].tolist()

In [6]:
df_AID = df_AID_base[df_AID_base["APT_ICAO"].isin(airports_code_list)].dropna()
#Isolating "adm" predictor
df_AID = (df_AID.groupby(["APT_ICAO", "Year_Lobt", "Month_Lobt"])
          .agg({
            "TF": "sum",
            "Total_Flights_Period": "first",
            "adm": "sum"
        }).reset_index())

df_AID["Delay_Ratio"] = (df_AID["TF"] / df_AID["Total_Flights_Period"]) * 100

In [7]:
consistency_ratio = df_AID.groupby(["APT_ICAO", "Year_Lobt"])["TF"].agg(["mean", "std"]).reset_index()
consistency_ratio["CR"] = consistency_ratio["std"]/consistency_ratio["mean"]

In [8]:
df = df_AID.merge(consistency_ratio[["APT_ICAO", "Year_Lobt", "CR"]], on=["APT_ICAO", "Year_Lobt"], how="left")
df.head()

,APT_ICAO,Year_Lobt,Month_Lobt,TF,Total_Flights_Period,adm,Delay_Ratio,CR
0,EDDF,2023,April,16336,12448,12.099695,131.233933,0.198526
1,EDDF,2023,August,21990,14135,20.627308,155.571277,0.198526
2,EDDF,2023,December,17029,10889,18.473322,156.387180,0.198526
3,EDDF,2023,February,11685,9206,15.133283,126.928090,0.198526
4,EDDF,2023,January,12250,10234,12.483877,119.699042,0.198526


In [10]:
df["Delay_Prone"] = np.where(df["Delay_Ratio"] >= 98, 1, 0)
# df.sort_values("Delay_Prone", ascending=True)

In [11]:
months = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
df["Month_Lobt"] = df["Month_Lobt"].map(months)


# Prediction

In [ ]:
features = ["Month_Lobt", "Total_Flights_Period", "adm", "CR"] # Model features
X = df[features] # predictors
y = df["Delay_Prone"] # predicted variable

numeric_features = ["Month_Lobt","Total_Flights_Period","adm", "CR"] # to be scaled

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features), # features to be scaled
    ]
)

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor), # Actual scaling
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=55, n_estimators=200)) # Model used, we use balanced weight to moderate class split 
])

#To avoid the model memorizing patterns rather than predict them, we train on 80% of the airports and test on the remaining 20%
unique_airports = df["APT_ICAO"].unique()
# For example, train on first 14 airports, test on remaining 6
train_airports = unique_airports[:16]
test_airports = unique_airports[16:]

X_train = df[df["APT_ICAO"].isin(train_airports)][features]
y_train = df[df["APT_ICAO"].isin(train_airports)]["Delay_Prone"]

X_test = df[df["APT_ICAO"].isin(test_airports)][features]
y_test = df[df["APT_ICAO"].isin(test_airports)]["Delay_Prone"]

# Fit and evaluate
rf_model.fit(X_train, y_train)# Model training

['LSZH' 'LTAI' 'LTFJ' 'LTFM']


,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred)) # (TP + TN)/(FP + FN)
print(confusion_matrix(y_test, y_pred)) # [TP, FP], [FN, TN]
print(classification_report(y_test, y_pred)) 

Accuracy: 0.8105263157894737
[[72  4]
 [14  5]]
              precision    recall  f1-score   support

           0       0.84      0.95      0.89        76
           1       0.56      0.26      0.36        19

    accuracy                           0.81        95
   macro avg       0.70      0.61      0.62        95
weighted avg       0.78      0.81      0.78        95



Let's calculate the average accuracy of the model across 100 random states to see how sensitive it is to them:

In [22]:
acc = []
unique_airports = df["APT_ICAO"].unique()
# For example, train on first 14 airports, test on remaining 6
train_airports = unique_airports[:16]
test_airports = unique_airports[16:]

X_train = df[df["APT_ICAO"].isin(train_airports)][features]
y_train = df[df["APT_ICAO"].isin(train_airports)]["Delay_Prone"]

X_test = df[df["APT_ICAO"].isin(test_airports)][features]
y_test = df[df["APT_ICAO"].isin(test_airports)]["Delay_Prone"]

for i in range(100):
    rf_model2 = Pipeline(steps=[
    ("preprocess", preprocessor), # Actual scaling
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=i, n_estimators=200)) # Model used, we use balanced weight to moderate class split 
    ])
    rf_model2.fit(X_train, y_train)# Model training
    y_pred = rf_model2.predict(X_test)
    acc.append(accuracy_score(y_test, y_pred))
print(np.mean(acc))

0.8103157894736844


In [23]:
importances = rf_model.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nFeature Importances:")
print(importance_df)


Feature Importances:
                Feature  Importance
2                   adm    0.461789
3                    CR    0.262379
1  Total_Flights_Period    0.228201
0            Month_Lobt    0.047631
